# DSA 8301 — Statistical Inference for Big Data
## Phase 2: Data Cleaning & Preparation
**Housing Financial Vulnerability Score (HFVS) | 2023/24 Kenya Housing Survey**
**Valerie Jerono | Reg. No. 222331 | Strathmore University iLabAfrica**

---

### What this notebook does
Takes `master_frame.parquet` (21,347 households × 443 columns) and produces
`master_frame_clean.parquet` — a fully analysis-ready frame for Phase 3 EDA and
subsequent HFVS modelling. Every decision here is **domain-informed** by the
Business & Data Understanding document (CRISP-DM Phase 1) and the Dataset
Reference Card.

**Pipeline (in order):**
1. Environment setup & load
2. KNBS sentinel-code decoding (−1, 98, 99, 996, 999 → NaN)
3. Defensive rename (raw KNBS codes → semantic names)
4. Full column audit (missingness, cardinality, module source)
5. Stage-1 structural drops (free-text, admin duplicates, constants, near-empty)
6. Multi-select indicator pruning (sparse <0.5% residual "other" sub-items)
7. Structural missingness encoding (rental k_, owned l_, land lp_, mortgage mort_)
8. Sentinel-aware group-median imputation (stratified by tenure type)
9. Monetary outlier treatment (Winsorize KES columns only)
10. Consistency checks & logical contradiction flags
11. **Feature engineering** — total expenditure, rent burden, crowding index,
    quality score, triple-exposure flag, county label
12. Pillar map integrity check + leakage assertion
13. Save cleaned frame + preprocessing summary table

> **Due date:** 17 June 2026


## 0 · Environment Setup

In [ ]:
# ── 0.1  Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
# ── 0.2  Install / upgrade dependencies (first run only) ─────────────────────
!pip install -q pyreadstat polars pyarrow scipy statsmodels
print('Dependencies ready.')


In [ ]:
# ── 0.3  Core imports ─────────────────────────────────────────────────────────
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, probplot
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')
np.random.seed(42)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 80)

plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': 'white',
    'axes.facecolor': '#F8F8F6', 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.titlesize': 13,
    'axes.titleweight': '600', 'axes.labelsize': 11,
    'xtick.labelsize': 9,  'ytick.labelsize': 9,
    'font.family': 'sans-serif', 'legend.fontsize': 9,
})

TEAL   = '#00695C'; RED    = '#B71C1C'; AMBER  = '#E65100'
BLUE   = '#1565C0'; PURPLE = '#6A1B9A'; GRAY   = '#546E7A'
DARK   = '#2C2C2A'; GREEN  = '#2E7D32'

print('All imports loaded.')


In [ ]:
# ── 0.4  Paths ────────────────────────────────────────────────────────────────
DRIVE = Path('/content/drive/MyDrive/KHS_Dissertation')
PQ    = DRIVE / 'data' / 'parquet'
FIGS  = DRIVE / 'outputs' / 'figures' / 'dsa8301'
TABS  = DRIVE / 'outputs' / 'tables'  / 'dsa8301'
for p in [FIGS, TABS]:
    p.mkdir(parents=True, exist_ok=True)

# ── 0.5  County map (1–47) ────────────────────────────────────────────────────
COUNTY_MAP = {
     1:'Mombasa',        2:'Kwale',          3:'Kilifi',         4:'Tana River',
     5:'Lamu',           6:'Taita-Taveta',   7:'Garissa',        8:'Wajir',
     9:'Mandera',       10:'Marsabit',      11:'Isiolo',        12:'Meru',
    13:'Tharaka-Nithi', 14:'Embu',          15:'Kitui',         16:'Machakos',
    17:'Makueni',       18:'Nyandarua',     19:'Nyeri',         20:'Kirinyaga',
    21:"Murang'a",      22:'Kiambu',        23:'Turkana',       24:'West Pokot',
    25:'Samburu',       26:'Trans Nzoia',   27:'Uasin Gishu',   28:'Elgeyo-Marakwet',
    29:'Nandi',         30:'Baringo',       31:'Laikipia',      32:'Nakuru',
    33:'Narok',         34:'Kajiado',       35:'Kericho',       36:'Bomet',
    37:'Kakamega',      38:'Vihiga',        39:'Bungoma',       40:'Busia',
    41:'Siaya',         42:'Kisumu',        43:'Homa Bay',      44:'Migori',
    45:'Kisii',         46:'Nyamira',       47:'Nairobi',
}
print(f'Paths ready.  |  FIGS → {FIGS}  |  TABS → {TABS}')


## 1 · Rename Map (KNBS codes → semantic names)

In [ ]:
# ── 1.1  Full column rename map ──────────────────────────────────────────────
# Covers every column expected in master_frame.parquet.
# Applied defensively: only renames columns actually present in df.

RENAME_MAP = {
    # Survey admin
    'interview__id'   : 'hh_uuid',
    'a12'             : 'interview_result',
    'tag'             : 'survey_tag',
    'rnd'             : 'sample_round',
    'selectedage'     : 'selected_respondent_age',

    # Identifiers & geography
    'interview__key'  : 'hh_id',
    'a01'             : 'county_code',
    'countycode'      : 'county_code_str',
    'a07_1'           : 'urban_rural',
    'serial'          : 'hh_serial',
    'hhweight'        : 'hh_weight',

    # Water & sanitation (core)
    'c01_1'  : 'water_src_main',
    'c01_2'  : 'water_collect_method',
    'c01_3'  : 'water_treated',
    'c01_4'  : 'water_dist_mins',
    'c01_5'  : 'water_src_main_season',
    'c02_1'  : 'water_src_secondary',
    'c02_2'  : 'water_collect_secondary',
    'c02_3'  : 'water_secondary_treated',
    'c02_4'  : 'water_secondary_dist_mins',
    'c02_5'  : 'water_secondary_season',
    'c03'    : 'n_toilet_facilities',
    'c04'    : 'toilet_type',
    'c05'    : 'has_handwash_facility',
    'c06'    : 'n_hh_sharing_toilet',
    'c07'    : 'handwash_materials',
    'c08'    : 'has_bathhouse',
    'c14_1'  : 'spend_water_kes',

    # Energy (core)
    'c10'    : 'lighting_src',
    'c10_2'  : 'electricity_hrs_day',
    'c10_3'  : 'electricity_supply_type',
    'c10_4'  : 'electricity_conn_type',
    'c11'    : 'cooking_fuel',
    'c11_1'  : 'cooking_fuel_n_types',
    'c11_2'  : 'cooking_fuel_secondary',
    'c11_2_1': 'cooking_fuel_secondary_spend_kes',
    'c11_3'  : 'cooking_fuel_tertiary',
    'c12'    : 'cooking_stove_type',
    'c12_1'  : 'n_cooking_spaces',
    'c12_2'  : 'cooking_space_indoor',
    'c12_3'  : 'cooking_stove_secondary',
    'c14_2'  : 'spend_electricity_kes',
    'c14_3'  : 'spend_energy_other_kes',

    # Assets
    'c13__1'  : 'owns_radio',
    'c13__2'  : 'owns_mobile',
    'c13__3'  : 'owns_tv',
    'c13__4'  : 'owns_computer',
    'c13__5'  : 'owns_motorcycle',
    'c13__6'  : 'owns_vehicle',
    'c13__7'  : 'owns_fridge',
    'c13__96' : 'owns_other_asset',
    'internet': 'has_internet',

    # Solid waste
    'c09__1' : 'waste_coll_county',
    'c09__2' : 'waste_coll_private',
    'c09__3' : 'waste_coll_ngo',
    'c09__4' : 'waste_coll_community',
    'c09__5' : 'waste_coll_self',
    'c09__6' : 'waste_coll_neighbour',
    'c09__7' : 'waste_coll_other',

    # Dwelling (household module)
    'd01'    : 'hh_tenure_type',
    'd17'    : 'dwelling_ownership_doc',
    'd18__1' : 'ownership_doc_title_deed',
    'd18__2' : 'ownership_doc_allotment',
    'd18__3' : 'ownership_doc_agreement',
    'd18__4' : 'ownership_doc_rent_receipt',
    'd18__5' : 'ownership_doc_will',
    'd18__6' : 'ownership_doc_letter',
    'd18__7' : 'ownership_doc_other',
    'd18__8' : 'ownership_doc_none',
    'd19'    : 'n_hh_in_building',
    'd20__1' : 'shared_facility_water',
    'd20__2' : 'shared_facility_toilet',
    'd20__3' : 'shared_facility_bathroom',
    'd20__4' : 'shared_facility_kitchen',
    'd20__5' : 'shared_facility_entrance',
    'd20__6' : 'shared_facility_yard',
    'd20__7' : 'shared_facility_parking',
    'd20__8' : 'shared_facility_none',
    'duration': 'tenancy_duration_cat',
    'year_occ': 'year_occupied_cat',
    'bf'      : 'building_floor_cat',

    # Income & expenditure (g01a–k)
    'g01a' : 'spend_food_kes',
    'g01b' : 'spend_clothing_kes',
    'g01c' : 'spend_education_kes',
    'g01d' : 'spend_health_kes',
    'g01e' : 'spend_transport_kes',
    'g01f' : 'spend_comms_kes',
    'g01g' : 'spend_recreation_kes',
    'g01h' : 'spend_housing_kes',
    'g01i' : 'spend_energy_kes',
    'g01j' : 'spend_other_kes',
    'g01k' : 'spend_remittances_kes',
    'g02'  : 'pays_rent',
    'g02_1': 'rent_monthly_kes',
    'g03'  : 'tenure_type',
    'g04'  : 'owns_other_property',

    # Housing problems (g05__)
    'g05__1'  : 'prob_overcrowding',
    'g05__2'  : 'prob_poor_water',
    'g05__3'  : 'prob_poor_sanitation',
    'g05__4'  : 'prob_poor_drainage',
    'g05__5'  : 'prob_poor_road',
    'g05__6'  : 'prob_insecurity',
    'g05__7'  : 'prob_high_cost',
    'g05__8'  : 'prob_poor_structure',
    'g05__9'  : 'prob_crime',
    'g05__10' : 'prob_other',

    # Housing aspirations (g06__)
    'g06__1' : 'aspire_buy_land',
    'g06__2' : 'aspire_build',
    'g06__3' : 'aspire_buy_house',
    'g06__4' : 'aspire_rent_better',
    'g06__5' : 'aspire_renovate',
    'g06__6' : 'aspire_move_county',
    'g06__7' : 'aspire_stay_improve',
    'g06__8' : 'aspire_no_change',

    # Housing perception (h01–h11)
    'h01' : 'perc_structure',
    'h02' : 'perc_roof',
    'h03' : 'perc_walls',
    'h04' : 'perc_floor',
    'h05' : 'perc_ventilation',
    'h06' : 'perc_lighting',
    'h07' : 'perc_water',
    'h08' : 'perc_sanitation',
    'h09' : 'perc_waste',
    'h10' : 'perc_security',
    'h11' : 'perc_overall',

    # Land ownership (i module)
    'i00' : 'owns_land',

    # Environment & hazards (e module)
    'e05' : 'dist_to_market_mins',
    'e06' : 'flood_exposure',
    'e07' : 'landslide_exposure',
    'e08' : 'other_hazard_exposure',
    'e09__1'  : 'hazard_lost_property',
    'e09__2'  : 'hazard_injured',
    'e09__3'  : 'hazard_displaced',
    'e09__4'  : 'hazard_death',
    'e09__5'  : 'hazard_none',

    # Tenure & mobility (j module — core)
    'j02'  : 'ever_owned_dwelling',
    'j04_1': 'is_owner_occupier',
    'j04_2': 'tenure_is_informal',
    'j05'  : 'has_title_doc',
    'j09'  : 'housing_cost_burden',
    'j10'  : 'missed_payment',
    'j11'  : 'eviction_risk',
    'j12_1': 'yrs_in_dwelling',
    'j13'  : 'satisfied_tenure',
    'j14'  : 'wants_to_own',
    'j15'  : 'applied_for_housing',
    'j17'  : 'willing_to_relocate',
    'j20'  : 'aware_affordable_housing',
    'j21'  : 'applied_affordable_housing',

    # Rental module (k — core)
    'k01'  : 'rental_market_type',
    'k02'  : 'has_written_lease',
    'k03'  : 'rent_negotiated',
    'k04'  : 'landlord_type',
    'k05'  : 'rent_actual_kes',
    'k09'  : 'lease_type',
    'k13'  : 'receipt_given',
    'k15'  : 'has_rent_dispute',
    'k20'  : 'plans_to_buy',
    'k21'  : 'rent_arrears',
    'k25'  : 'willingness_to_pay_kes',
    'min_rent': 'psu_min_rent_kes',

    # Owned/self-built dwelling (l module — core)
    'l07'  : 'dwelling_yr_surveyed',
    'l08'  : 'dwelling_yr_built',
    'l13'  : 'dwelling_value_kes',
    'l14'  : 'land_value_kes',
    'l15'  : 'mortgage_repayment_kes',
    'l19'  : 'plot_size_decimals',
    'l21'  : 'had_renovation',
    'l28'  : 'yr_last_renovated',

    # Derived / computed (KNBS pre-computed)
    'prop_util'  : 'util_income_ratio',
    'med_prop'   : 'cty_med_util_ratio',
    'utilities'  : 'pays_utilities',
    'ctymin_ut'  : 'cty_min_utility_kes',
    'med_brms'   : 'cty_med_bedrooms',
    'sf'         : 'is_slum',
    'pln'        : 'settlement_plan_status',

    # Dwelling aggregates (from dwelling file)
    'dw_type'              : 'dw_type',
    'dw_wall_material'     : 'dw_wall_mat',
    'dw_roof_material'     : 'dw_roof_mat',
    'dw_floor_material'    : 'dw_floor_mat',
    'dw_n_rooms'           : 'dw_rooms',
    'dw_floor_area_m2'     : 'dw_area_m2',
    'dw_n_bedrooms'        : 'dw_bedrooms',
    'dw_approved'          : 'dw_approved',
    'dw_planning_ok'       : 'dw_has_planning',
    'dw_hazard_zone'       : 'dw_in_hazard_zone',
    'dw_n_units_enumerated': 'dw_units_count',

    # Individual aggregates
    'hh_size'        : 'hh_size',
    'hhh_sex'        : 'hh_head_sex',
    'any_disability' : 'has_disability',
    'max_edu_isced'  : 'max_edu_isced',
    'mean_age'       : 'mean_age',
    'dependency_ratio': 'dependency_ratio',
    'n_female'       : 'n_female',
    'n_children'     : 'n_children',
    'n_elderly'      : 'n_elderly',
    'n_working_age'  : 'n_working_age',

    # Land parcel aggregates (55% structural missing — i00==0 households)
    'lp_n_parcels'     : 'lp_n_parcels',
    'lp_primary_tenure': 'lp_tenure_type',
    'lp_has_title'     : 'lp_has_title',
    'lp_primary_use'   : 'lp_land_use',
    'lp_any_dispute'   : 'lp_has_dispute',
    'lp_any_registered': 'lp_is_registered',
    'lp_any_collateral': 'lp_used_as_collateral',

    # County aggregates
    'cty_housing_stock'           : 'cty_housing_stock',
    'cty_housing_backlog'         : 'cty_housing_backlog',
    'cty_planning_staff'          : 'cty_planning_staff',
    'cty_has_housing_policy'      : 'cty_has_housing_policy',
    'cty_building_approval_system': 'cty_approval_system',

    # NEMA aggregates
    'nema_eia_applications': 'nema_eia_apps',
    'nema_eia_approvals'   : 'nema_eia_approvals',
    'nema_processing_days' : 'nema_proc_days',

    # Water service aggregates
    'wsvc_water_connections': 'wsvc_water_conns',
    'wsvc_sewer_connections' : 'wsvc_sewer_conns',
    'wsvc_water_tariff'      : 'wsvc_tariff',
    'wsvc_service_quality'   : 'wsvc_quality',
    'wsvc_n_providers'       : 'wsvc_n_providers',

    # Mortgage aggregates (60–81% structural missing — county coverage)
    'mort_interest_rate' : 'mort_rate',
    'mort_ltv_ratio'     : 'mort_ltv',
    'mort_avg_term_years': 'mort_term_yrs',
    'mort_n_providers'   : 'mort_n_providers',

    # Loan aggregates
    'loan_avg_size'    : 'loan_avg_kes',
    'loan_outstanding' : 'loan_outstanding_kes',
    'loan_n_providers' : 'loan_n_providers',

    # Financier aggregates
    'fin_portfolio'        : 'fin_portfolio_kes',
    'fin_avg_tenure_months': 'fin_tenure_months',
    'fin_n_financiers'     : 'fin_n_financiers',
}
print(f'RENAME_MAP defined — {len(RENAME_MAP)} entries.')


## 2 · Load & First-Look

In [ ]:
# ── 2.1  Load master_frame.parquet ───────────────────────────────────────────
df = pd.read_parquet(PQ / 'master_frame.parquet')
print(f'Loaded  → {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory  → {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()
print('First 5 columns:', df.columns[:5].tolist())
print('Last  5 columns:', df.columns[-5:].tolist())


## 3 · Sentinel-Code Decoding

**Critical first step before any imputation.**

KNBS SurveySolutions exports use several numeric sentinel codes that must be
decoded to NaN so they are not treated as real values during imputation or
outlier analysis:

| Code | Meaning |
|------|---------|
| `−1` | Not applicable / skipped |
| `98` / `998` | Don't know |
| `99` / `999` | Refused / missing |
| `96` / `996` | Other (specify) — open-ended |

These codes are identical to genuine answers in some variables (e.g. county code 96
does not exist — safe to recode; but age 98 is not impossible — this is handled
per-column for sensitive variables).

In [ ]:
# ── 3.1  Decode KNBS sentinels to NaN ────────────────────────────────────────
# Only recode numeric columns; strings are already "Unknown" or filled.
# Protect: county_code (1–47 valid), hh_weight, any binary 0/1 column.

PROTECTED_FROM_SENTINEL = {
    'county_code', 'hh_weight', 'urban_rural', 'hh_id',
    # binary ownership / flag cols — 0 and 1 are real values
    'owns_radio', 'owns_mobile', 'owns_tv', 'owns_computer',
    'owns_motorcycle', 'owns_vehicle', 'owns_fridge', 'has_internet',
    'prob_overcrowding', 'prob_poor_water', 'prob_poor_sanitation',
    'prob_poor_drainage', 'prob_poor_road', 'prob_insecurity',
    'prob_high_cost', 'prob_poor_structure', 'prob_crime',
    'aspire_buy_land', 'aspire_build', 'aspire_buy_house',
    'aspire_rent_better', 'aspire_renovate', 'aspire_move_county',
    'aspire_stay_improve', 'aspire_no_change',
    'ownership_doc_title_deed', 'ownership_doc_allotment',
    'ownership_doc_agreement', 'ownership_doc_rent_receipt',
    'ownership_doc_will', 'ownership_doc_letter', 'ownership_doc_none',
    'shared_facility_water', 'shared_facility_toilet',
    'shared_facility_bathroom', 'shared_facility_kitchen',
    'shared_facility_entrance', 'shared_facility_yard',
    'shared_facility_parking', 'shared_facility_none',
    'waste_coll_county', 'waste_coll_private', 'waste_coll_ngo',
    'waste_coll_community', 'waste_coll_self', 'waste_coll_neighbour',
    'dw_approved', 'dw_has_planning', 'dw_in_hazard_zone',
    'has_disability', 'lp_has_title', 'lp_has_dispute',
    'lp_is_registered', 'lp_used_as_collateral',
    'cty_has_housing_policy',
}

# First apply rename so we're working with semantic names
rename_now = {raw: clean for raw, clean in RENAME_MAP.items() if raw in df.columns}
df = df.rename(columns=rename_now)
INVERSE_MAP = {clean: orig for orig, clean in RENAME_MAP.items()}
print(f'Applied {len(rename_now)} renames.')

# Sentinel decoding
SENTINELS_GENERAL = {-1, 98, 99, 96}
SENTINELS_THREEDIGIT = {998, 999, 996, 997}

sentinel_log = []
numeric_cols = df.select_dtypes(include='number').columns

for c in numeric_cols:
    if c in PROTECTED_FROM_SENTINEL:
        continue
    n_before = df[c].isna().sum()
    # Decode -1, 98, 99, 96, 998, 999, 996, 997
    mask = df[c].isin(SENTINELS_GENERAL | SENTINELS_THREEDIGIT)
    n_flagged = mask.sum()
    if n_flagged > 0:
        df.loc[mask, c] = np.nan
        sentinel_log.append((c, n_flagged))

sentinel_df = pd.DataFrame(sentinel_log, columns=['column', 'n_decoded_to_nan'])
print(f'Columns with sentinel values decoded: {len(sentinel_df)}')
print(f'Total cells decoded to NaN: {sentinel_df["n_decoded_to_nan"].sum():,}')
sentinel_df.sort_values('n_decoded_to_nan', ascending=False).head(20)


## 4 · Column Audit

In [ ]:
# ── 4.1  Full column audit: missingness, cardinality, source module ──────────

def get_module(clean_name):
    orig = INVERSE_MAP.get(clean_name, clean_name)
    m = re.match(r'^([a-z]+)\d', orig)
    if m:
        return m.group(1)
    return orig.split('_')[0]

MODULE_LABELS = {
    'a' : 'admin_geo',            'b' : 'individual_raw',
    'c' : 'water_energy_assets',  'd' : 'dwelling_hh_module',
    'e' : 'environment_hazards',  'g' : 'income_expenditure',
    'h' : 'housing_perception',   'i' : 'land_ownership',
    'j' : 'tenure_mobility',      'k' : 'rental_module',
    'l' : 'owned_dwelling',
}

def dominant_share(s):
    vc = s.value_counts(normalize=True, dropna=True)
    return round(float(vc.iloc[0]) * 100, 2) if len(vc) else np.nan

audit = pd.DataFrame(index=df.columns)
audit['orig_code']    = [INVERSE_MAP.get(c, c) for c in df.columns]
audit['module']       = [get_module(c) for c in df.columns]
audit['module_label'] = audit['module'].map(MODULE_LABELS).fillna('derived_aggregate')
audit['dtype']        = df.dtypes.astype(str)
audit['pct_missing']  = (df.isna().mean() * 100).round(2)
audit['n_unique']     = df.nunique(dropna=True)
audit['dominant_pct'] = [dominant_share(df[c]) for c in df.columns]

print('=== Columns per module ===')
print(audit['module_label'].value_counts().to_string())
print()
print(f'Columns with  >50% missing : {(audit["pct_missing"] >  50).sum()}')
print(f'Columns with  >80% missing : {(audit["pct_missing"] >  80).sum()}')
print(f'Constant columns (≤1 value): {(audit["n_unique"] <= 1).sum()}')

# Missingness tier classification (per Business Understanding §6)
def missingness_tier(pct):
    if pct == 0:      return 'Complete (0%)'
    if pct <= 20:     return 'Low (1–20%)'
    if pct <= 60:     return 'Moderate (21–60%)'
    return             'High (>60%)'

audit['miss_tier'] = audit['pct_missing'].apply(missingness_tier)
print()
print('=== Missingness tier distribution ===')
print(audit['miss_tier'].value_counts().to_string())


## 5 · Stage-1 Structural Drops

Four drop categories, with a protected core set immune to auto-drop.
The core set includes every Pillar-ingredient variable, every proxy-feature
candidate, and all derived aggregate blocks.

In [ ]:
# ── 5.1  Core variables — protected from auto-drop ───────────────────────────
CORE_KEEP = {
    # Geography / weights
    'hh_id', 'county_code', 'urban_rural', 'hh_weight',
    # Water & sanitation (Pillar 2 + 5 ingredients)
    'water_src_main', 'water_collect_method', 'water_treated', 'water_dist_mins',
    'water_src_secondary', 'toilet_type', 'has_handwash_facility',
    'handwash_materials', 'spend_water_kes', 'n_toilet_facilities',
    'n_hh_sharing_toilet',
    # Energy (Pillar 5 ingredients)
    'lighting_src', 'electricity_hrs_day', 'electricity_conn_type',
    'cooking_fuel', 'cooking_stove_type', 'spend_electricity_kes',
    'spend_energy_other_kes',
    # Assets (proxy features)
    'owns_radio', 'owns_mobile', 'owns_tv', 'owns_computer',
    'owns_motorcycle', 'owns_vehicle', 'owns_fridge', 'has_internet',
    # Income & expenditure (Pillar 1 ingredients — all 11 spend columns)
    'spend_food_kes', 'spend_clothing_kes', 'spend_education_kes',
    'spend_health_kes', 'spend_transport_kes', 'spend_comms_kes',
    'spend_recreation_kes', 'spend_housing_kes', 'spend_energy_kes',
    'spend_other_kes', 'spend_remittances_kes',
    'pays_rent', 'rent_monthly_kes', 'tenure_type', 'owns_other_property',
    # Housing problems (g05)
    'prob_overcrowding', 'prob_poor_water', 'prob_poor_sanitation',
    'prob_poor_drainage', 'prob_poor_road', 'prob_insecurity',
    'prob_high_cost', 'prob_poor_structure',
    # Housing perception (h01–h11) — Pillar 2 ingredients
    'perc_structure', 'perc_roof', 'perc_walls', 'perc_floor',
    'perc_ventilation', 'perc_lighting', 'perc_water', 'perc_sanitation',
    'perc_waste', 'perc_security', 'perc_overall',
    # Tenure & mobility (Pillar 3 ingredients)
    'is_owner_occupier', 'has_title_doc', 'housing_cost_burden',
    'missed_payment', 'eviction_risk', 'yrs_in_dwelling', 'satisfied_tenure',
    'hh_tenure_type', 'tenure_is_informal', 'ever_owned_dwelling',
    'n_hh_in_building', 'tenancy_duration_cat', 'year_occupied_cat',
    'building_floor_cat',
    # Environment & hazards (Pillar 4 ingredients)
    'flood_exposure', 'landslide_exposure', 'other_hazard_exposure',
    'dist_to_market_mins',
    'hazard_lost_property', 'hazard_injured', 'hazard_displaced',
    'hazard_death', 'hazard_none',
    # Rental module (core — Pillar 1 & 3)
    'has_written_lease', 'rent_actual_kes', 'lease_type', 'rent_arrears',
    'psu_min_rent_kes', 'plans_to_buy', 'rental_market_type', 'rent_negotiated',
    # Owned-dwelling module (core — Pillar 1 & 2)
    'dwelling_yr_built', 'dwelling_yr_surveyed', 'dwelling_value_kes',
    'land_value_kes', 'mortgage_repayment_kes', 'plot_size_decimals',
    'had_renovation', 'yr_last_renovated',
    # Land ownership (Pillar 3 ingredients)
    'owns_land',
    # Derived household indicators
    'util_income_ratio', 'pays_utilities', 'cty_med_util_ratio',
    'cty_min_utility_kes', 'cty_med_bedrooms', 'is_slum',
    'settlement_plan_status',
    # Housing aspirations (proxy features)
    'aspire_buy_land', 'aspire_build', 'aspire_buy_house',
    'aspire_rent_better', 'aspire_renovate', 'aspire_move_county',
    'aspire_stay_improve', 'aspire_no_change',
    'wants_to_own', 'willing_to_relocate', 'aware_affordable_housing',
    # Shared facility indicators
    'shared_facility_water', 'shared_facility_toilet',
    'shared_facility_bathroom', 'shared_facility_kitchen', 'shared_facility_none',
    # Waste collection
    'waste_coll_county', 'waste_coll_private', 'waste_coll_community',
    'waste_coll_self',
}

# Protect all derived aggregate blocks wholesale
CORE_KEEP |= set(audit.index[audit['module_label'] == 'derived_aggregate'])
# Keep only columns that actually exist
CORE_KEEP &= set(df.columns)

print(f'Core variables protected from auto-drop: {len(CORE_KEEP)}')


In [ ]:
# ── 5.2  Apply stage-1 structural drops ──────────────────────────────────────

# (a) Free-text "other / specify / text" fields
FREE_TEXT_RX = re.compile(
    r'(other_text|other_specify|_other\d*$|_other$|_text$|_specify$)', re.I
)
free_text_cols = [c for c in df.columns
                  if FREE_TEXT_RX.search(c) and c not in CORE_KEEP]

# (b) Admin / non-analytical identifiers
admin_drop = [c for c in [
    'survey_tag', 'sample_round', 'selected_respondent_age',
    'county_code_str', 'hh_serial', 'interview_result', 'hh_uuid',
] if c in df.columns]

if {'county_code', 'county_code_str'}.issubset(df.columns):
    agree = (df['county_code'].astype(str).str.zfill(2)
             == df['county_code_str'].astype(str).str.zfill(2)).mean()
    print(f'county_code vs county_code_str agreement: {agree:.2%}  → dropping county_code_str')

# (c) Constant columns (zero variance)
constant_cols = [c for c in df.columns
                 if df[c].nunique(dropna=True) <= 1 and c not in CORE_KEEP]

# (d) Near-empty columns (>80% missing) outside protected set
near_empty_cols = audit.index[
    (audit['pct_missing'] > 80) & (~audit.index.isin(CORE_KEEP))
].tolist()

stage1_drop = sorted(set(free_text_cols + admin_drop + constant_cols + near_empty_cols))

print(f'Free-text / specify fields        : {len(free_text_cols)}')
print(f'Admin duplicates                  : {len(admin_drop)}')
print(f'Constant columns                  : {len(constant_cols)}')
print(f'Near-empty (>80% missing)         : {len(near_empty_cols)}')
print(f'TOTAL stage-1 drops               : {len(stage1_drop)}')

df = df.drop(columns=stage1_drop)
print(f'\nShape after stage 1: {df.shape}')


## 6 · Multi-Select Indicator Pruning

In [ ]:
# ── 6.1  Prune sparse multi-select sub-items ─────────────────────────────────
# Only drop residual "other" tail options selected by <0.5% of households.
# Substantive indicators (even rare ones) are retained — rarity is informative.

multiselect_groups = {}
for c in df.columns:
    orig = INVERSE_MAP.get(c, '')
    m = re.match(r'^([a-z0-9]+)__\d+', orig)
    if m:
        multiselect_groups.setdefault(m.group(1), []).append(c)

print(f'Multi-select question groups found: {len(multiselect_groups)}')

ms_prune = []
for grp, cols in multiselect_groups.items():
    for c in cols:
        if c in CORE_KEEP:
            continue
        rate = df[c].mean(skipna=True)
        if pd.notna(rate) and rate < 0.005:
            ms_prune.append((grp, c, round(rate * 100, 3)))

ms_prune_df = pd.DataFrame(ms_prune, columns=['group', 'column', 'selected_pct'])
print(f'Sparse sub-items (<0.5% selected): {len(ms_prune_df)}')

df = df.drop(columns=ms_prune_df['column'].tolist())
print(f'Shape after multi-select pruning: {df.shape}')


## 7 · Structural Missingness Encoding

This is the most domain-critical step. Three blocks of columns have high
missingness that is **not random** — it encodes "not applicable":

| Block | Miss% | Reason |
|-------|-------|--------|
| Rental `k_*` | ~68% | Only asked of tenant households |
| Owned `l_*` | ~55% | Only asked of owner-occupier households |
| Land parcel `lp_*` | ~55% | Only households where `i00==1` (own land) |
| Mortgage `mort_rate` | ~81% | Counties with no formal mortgage market |

**Treatment:**
- Create `is_renter` and `is_owner` flags first
- Categorical/ordinal k_/l_ columns → recode NaN to **−1** ("Not applicable")
- Continuous KES columns in k_/l_ → **leave as NaN** (excluded per tenure group)
- `lp_*` columns → NaN for `i00==0` households = **zero score contribution**
  (not imputed — encodes absence of land as the tenure condition it is)
- `mort_rate` → NaN = **no formal mortgage market** (encoded as maximum
  exclusion, not imputed)

In [ ]:
# ── 7.1  Tenure flags ────────────────────────────────────────────────────────
# is_renter: pays_rent == 1 (g02 original)
# is_owner:  tenure_type == 1 (g03 original; owner-occupier)
# Note: these are not mutually exclusive — employer-provided housing is neither

if 'pays_rent' in df.columns:
    df['is_renter'] = df['pays_rent'].eq(1).astype(int)
    n_renters = df['is_renter'].sum()
    print(f'is_renter=1 : {n_renters:,} ({n_renters/len(df):.1%})')

if 'tenure_type' in df.columns:
    df['is_owner'] = df['tenure_type'].eq(1).astype(int)
    n_owners = df['is_owner'].sum()
    print(f'is_owner=1  : {n_owners:,} ({n_owners/len(df):.1%})')

# Households that are neither renter nor owner (employer-provided, informal etc.)
if {'is_renter', 'is_owner'}.issubset(df.columns):
    n_other = ((df['is_renter'] == 0) & (df['is_owner'] == 0)).sum()
    print(f'Neither (employer/informal/other): {n_other:,} ({n_other/len(df):.1%})')


In [ ]:
# ── 7.2  Recode k_* and l_* structural NaNs ──────────────────────────────────
# Categorical/ordinal (≤12 unique values) → -1 = "Not applicable"
# Continuous KES → leave NaN (excluded per subpopulation in analysis)

k_cols = [c for c in df.columns if get_module(c) == 'k']
l_cols = [c for c in df.columns if get_module(c) == 'l']

KES_CONTINUOUS = {  # KES variables that stay NaN (subpopulation-specific)
    'rent_actual_kes', 'rent_monthly_kes', 'psu_min_rent_kes',
    'dwelling_value_kes', 'land_value_kes', 'mortgage_repayment_kes',
    'willingness_to_pay_kes',
}

struct_recode_log = []
for c in k_cols + l_cols:
    if c not in df.columns or c in KES_CONTINUOUS:
        continue
    n_missing = df[c].isna().sum()
    if n_missing == 0:
        continue
    if pd.api.types.is_numeric_dtype(df[c]) and df[c].dropna().nunique() <= 15:
        df[c] = df[c].fillna(-1)    # -1 = Not applicable
        struct_recode_log.append((c, 'recoded -1 (N/A)', n_missing))
    else:
        struct_recode_log.append((c, 'left NaN (continuous KES)', n_missing))

struct_df = pd.DataFrame(struct_recode_log,
                         columns=['column', 'treatment', 'n_structural_missing'])
print(f'k_/l_ structural-missingness treatments: {len(struct_df)}')
print(struct_df.to_string(index=False))


In [ ]:
# ── 7.3  Land parcel (lp_*) — encode absence as zero contribution ────────────
# For households where owns_land==0, lp_* values are NaN.
# Encode as 0 for count/binary columns (n_parcels=0, has_title=0, etc.)
# This means: no land → zero score on land-related tenure security indicators.
# Do NOT impute; zero is the correct domain value for landless households.

LP_BINARY_COLS = [c for c in df.columns
                  if c.startswith('lp_') and c not in ['lp_n_parcels', 'lp_land_use', 'lp_tenure_type']]
LP_COUNT_COLS  = ['lp_n_parcels']

if 'owns_land' in df.columns:
    landless_mask = df['owns_land'].eq(0)
    for c in LP_BINARY_COLS:
        if c in df.columns:
            df.loc[landless_mask, c] = df.loc[landless_mask, c].fillna(0)
    for c in LP_COUNT_COLS:
        if c in df.columns:
            df.loc[landless_mask, c] = df.loc[landless_mask, c].fillna(0)
    print(f'Landless households (i00==0): {landless_mask.sum():,}')
    print(f'lp_* binary cols zero-filled for landless: {len(LP_BINARY_COLS)}')
    print(f'lp_* count cols zero-filled for landless: {len(LP_COUNT_COLS)}')
    # lp_land_use and lp_tenure_type for landless → -1 (Not applicable)
    for c in ['lp_land_use', 'lp_tenure_type']:
        if c in df.columns:
            df.loc[landless_mask, c] = df.loc[landless_mask, c].fillna(-1)


In [ ]:
# ── 7.4  mort_rate — no formal mortgage market → flag, not impute ────────────
# 81% missing in mort_rate = county has no formal mortgage institution.
# Null is the data. Encode a binary exclusion flag alongside the column.

if 'mort_rate' in df.columns:
    df['mort_no_market'] = df['mort_rate'].isna().astype(int)
    n_no_mkt = df['mort_no_market'].sum()
    print(f'Counties with no formal mortgage market (mort_rate is NaN): ',
          f'{n_no_mkt:,} households ({n_no_mkt/len(df):.1%})')
    # mort_rate itself stays NaN — do not impute


## 8 · Group-Median Imputation (Stratified by Tenure Type)

After structural missingness is correctly encoded, remaining missing values
are imputed following the missingness tier protocol (Business Understanding §6):

- **Categorical / ordinal (≤10 unique values):** mode within tenure group
- **Continuous:** group-median within tenure group  
  *(group = tenure_type: owner / renter / other)*
- **String / object:** constant `"Unknown"`

An `_imputed` flag is created for every continuous column that needed imputation
(for audit and for downstream model diagnostics).

In [ ]:
# ── 8.1  Define tenure stratification groups ─────────────────────────────────
# Impute within three strata to respect the different missing patterns
# between owner-occupiers (k_ vars all -1) and renters (l_ vars all -1).

def tenure_group(row):
    if 'is_renter' in row and row['is_renter'] == 1:
        return 'renter'
    if 'is_owner' in row and row['is_owner'] == 1:
        return 'owner'
    return 'other'

if 'is_renter' in df.columns and 'is_owner' in df.columns:
    df['_tenure_group'] = df.apply(tenure_group, axis=1)
    print('Tenure group distribution:')
    print(df['_tenure_group'].value_counts().to_string())
else:
    df['_tenure_group'] = 'all'
    print('Tenure flag columns not found — imputing globally.')


In [ ]:
# ── 8.2  Impute remaining missing values ─────────────────────────────────────

remaining_missing = df.isna().mean() * 100
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)

# Skip structural KES columns — they stay NaN by design
SKIP_IMPUTE = KES_CONTINUOUS | {'mort_rate', 'util_income_ratio',
                                  'cty_med_util_ratio', 'psu_min_rent_kes'}
remaining_missing = remaining_missing[~remaining_missing.index.isin(SKIP_IMPUTE)]

print(f'Columns needing imputation (after structural encoding): {len(remaining_missing)}')
print(remaining_missing.head(25).to_string())


In [ ]:
# ── 8.3  Run group-stratified imputation ─────────────────────────────────────

impute_log = []
groups = df['_tenure_group'].unique()

for c in remaining_missing.index:
    if c not in df.columns:
        continue
    pct = remaining_missing[c]
    is_numeric = pd.api.types.is_numeric_dtype(df[c])

    if is_numeric:
        n_unique = df[c].dropna().nunique()
        if n_unique <= 10:
            method = 'group_mode'
            for g in groups:
                mask = df['_tenure_group'] == g
                mode_val = df.loc[mask, c].mode(dropna=True)
                fill_val = mode_val.iloc[0] if len(mode_val) else df[c].mode(dropna=True).iloc[0]
                df.loc[mask & df[c].isna(), c] = fill_val
            fill_display = f'group_mode (fallback global)'
        else:
            method = 'group_median'
            # Create imputation flag for continuous columns
            flag_col = f'{c}_imputed'
            df[flag_col] = df[c].isna().astype(int)
            for g in groups:
                mask = df['_tenure_group'] == g
                med = df.loc[mask, c].median()
                if pd.isna(med):
                    med = df[c].median()
                df.loc[mask & df[c].isna(), c] = med
            fill_display = f'group_median (+ {flag_col} flag)'
        impute_log.append((c, method, round(pct, 2)))
    else:
        df[c] = df[c].fillna('Unknown')
        impute_log.append((c, 'constant_Unknown', round(pct, 2)))

# Drop the helper tenure-group column
df = df.drop(columns=['_tenure_group'])

impute_log_df = pd.DataFrame(impute_log, columns=['column', 'method', 'pct_missing_before'])
print(f'\nImputation complete. Columns imputed: {len(impute_log_df)}')
impute_log_df


## 9 · Outlier Treatment — KES Monetary Variables

Only monetary (KES-denominated) columns are Winsorized. Perception scales
(h01–h11, Likert 1–3) are range-checked separately. Non-monetary numeric
columns are not capped — their distributions are examined in Phase 3 EDA.

In [ ]:
# ── 9.1  Perception scale range check (h01–h11 should be 1–3) ───────────────
perc_cols = [c for c in df.columns if c.startswith('perc_')]
range_log = []
for c in perc_cols:
    if c not in df.columns:
        continue
    out_of_range = ~df[c].isin([1, 2, 3]) & df[c].notna()
    n_bad = out_of_range.sum()
    if n_bad > 0:
        print(f'  {c}: {n_bad} values outside {{1,2,3}} → set to NaN')
        df.loc[out_of_range, c] = np.nan
        range_log.append((c, n_bad))
if not range_log:
    print('All perception columns (h01–h11) within valid range [1, 2, 3]. ✓')


In [ ]:
# ── 9.2  Winsorize KES monetary columns ──────────────────────────────────────
# Clip negatives to 0 first; then cap at 1st/99th percentile of NON-ZERO values.
# Non-zero floor is crucial: valid zero expenditure rows must not be pulled up.

money_cols = [c for c in df.columns if c.endswith('_kes') and c not in KES_CONTINUOUS]
print(f'Monetary columns to Winsorize: {len(money_cols)}')

outlier_log = []
for c in money_cols:
    if c not in df.columns:
        continue
    n_neg = (df[c] < 0).sum()
    if n_neg > 0:
        df[c] = df[c].clip(lower=0)

    nonzero = df.loc[df[c] > 0, c].dropna()
    if len(nonzero) < 30:
        continue
    lo, hi = nonzero.quantile(0.01), nonzero.quantile(0.99)
    n_hi = (df[c] > hi).sum()
    n_lo = ((df[c] < lo) & (df[c] > 0)).sum()
    df[c] = df[c].clip(upper=hi)   # only cap high; low zero stays zero
    outlier_log.append((c, n_neg, n_lo, n_hi, round(float(lo), 0), round(float(hi), 0)))

outlier_df = pd.DataFrame(outlier_log,
    columns=['column', 'neg_clipped', 'low_outliers', 'high_outliers', 'p1', 'p99'])
print(f'Winsorized columns: {len(outlier_df)}')
outlier_df


## 10 · Consistency Checks & Logical Contradictions

In [ ]:
# ── 10.1  Duplicate household IDs ────────────────────────────────────────────
if 'hh_id' in df.columns:
    n_dup = df['hh_id'].duplicated().sum()
    print(f'Duplicate hh_id rows: {n_dup}')
    if n_dup > 0:
        print('  → Keeping first occurrence, dropping duplicates.')
        df = df.drop_duplicates(subset='hh_id', keep='first')

# ── 10.2  County code validity ────────────────────────────────────────────────
if 'county_code' in df.columns:
    bad_county = ~df['county_code'].between(1, 47)
    print(f'county_code outside 1–47: {bad_county.sum()}')
    if bad_county.sum() > 0:
        df.loc[bad_county, 'county_code'] = np.nan

# ── 10.3  urban_rural coding (KNBS: 1=Urban, 2=Rural) ────────────────────────
if 'urban_rural' in df.columns:
    vc = df['urban_rural'].value_counts()
    print(f'urban_rural value counts (1=Urban, 2=Rural):')
    print(vc.to_string())
    unexpected = ~df['urban_rural'].isin([1, 2]) & df['urban_rural'].notna()
    if unexpected.sum() > 0:
        print(f'  WARNING: {unexpected.sum()} values outside {{1,2}}')

# ── 10.4  Dwelling year built validity ────────────────────────────────────────
if 'dwelling_yr_built' in df.columns:
    bad_yr = ~df['dwelling_yr_built'].between(1900, 2026) & df['dwelling_yr_built'].notna()
    print(f'dwelling_yr_built outside 1900–2026: {bad_yr.sum()} → set to NaN')
    df.loc[bad_yr, 'dwelling_yr_built'] = np.nan

# ── 10.5  Household size validity ─────────────────────────────────────────────
if 'hh_size' in df.columns:
    bad_sz = ~df['hh_size'].between(1, 50)
    print(f'hh_size outside 1–50: {bad_sz.sum()}')

# ── 10.6  Logical contradiction: is_owner=1 AND rent_actual_kes > 0 ───────────
if {'is_owner', 'rent_actual_kes'}.issubset(df.columns):
    contra_own_rent = (df['is_owner'] == 1) & (df['rent_actual_kes'] > 0)
    print(f'Owner-occupiers with rent_actual_kes > 0 (flag, not altered): '
          f'{contra_own_rent.sum()}')
    df['flag_owner_with_rent'] = contra_own_rent.astype(int)

# ── 10.7  Rent burden validation: pays_rent=0 but rent_actual_kes > 0 ─────────
if {'pays_rent', 'rent_actual_kes'}.issubset(df.columns):
    contra_rent = (df['pays_rent'] != 1) & (df['rent_actual_kes'] > 0)
    print(f'Non-renters (pays_rent≠1) with rent_actual_kes > 0: {contra_rent.sum()} (flagged)')
    df['flag_nonrenter_with_rent'] = contra_rent.astype(int)

print()
print(f'Shape after consistency checks: {df.shape}')


## 11 · Feature Engineering

These are the **analytical features** that feed directly into HFVS pillar
construction. They are computed here — in Phase 2 — so Phase 3 EDA works
with real pillar-relevant metrics rather than only raw survey codes.

| Feature | Formula | Pillar |
|---------|---------|--------|
| `total_exp` | Σ(g01a … g01k) | 1 — Financial Stress |
| `rent_burden_ratio` | k05 / (total_exp − g01h) | 1 |
| `rent_burdened` | rent_burden > 0.30 (SDG 11) | 1 |
| `persons_per_room` | hh_size / dw_rooms | 2 — Physical Quality |
| `dw_age_yrs` | 2024 − dwelling_yr_built | 2 |
| `obj_quality_score` | Normalised wall+roof+floor materials | 2 |
| `triple_exposed` | flood AND informal AND no_title | 4 — Hazard |
| `county_name` | COUNTY_MAP lookup | Geography |
| `util_burden_ratio` | (spend_water+electricity) / total_exp | 5 — Utility Deprivation |

In [ ]:
# ── 11.1  Total household expenditure (income proxy) ─────────────────────────
# World Bank LSMS / KIHBS convention: sum of 11 monthly expenditure categories.
# All 11 g01a–k columns have 0% base missingness — safe to sum.

EXP_COLS = [
    'spend_food_kes', 'spend_clothing_kes', 'spend_education_kes',
    'spend_health_kes', 'spend_transport_kes', 'spend_comms_kes',
    'spend_recreation_kes', 'spend_housing_kes', 'spend_energy_kes',
    'spend_other_kes', 'spend_remittances_kes',
]
present_exp = [c for c in EXP_COLS if c in df.columns]

if len(present_exp) >= 8:  # at least 8 of 11 needed for a reliable sum
    df['total_exp'] = df[present_exp].sum(axis=1, min_count=8)
    print(f'total_exp constructed from {len(present_exp)} expenditure components.')
    print(f'  Median total_exp: KES {df["total_exp"].median():,.0f}/month')
    print(f'  Mean   total_exp: KES {df["total_exp"].mean():,.0f}/month')
    print(f'  Missing total_exp: {df["total_exp"].isna().sum()}')
else:
    print(f'WARNING: only {len(present_exp)} expenditure columns found. total_exp not constructed.')


In [ ]:
# ── 11.2  Rent burden ratio (renter subpopulation only) ──────────────────────
# rent_burden = rent_actual_kes / (total_exp - spend_housing_kes)
# Denominator subtracts spend_housing_kes to avoid double-counting.
# rent_burdened = rent_burden > 0.30 (SDG 11 / KIHBS 30% threshold)

if {'rent_actual_kes', 'total_exp', 'spend_housing_kes', 'is_renter'}.issubset(df.columns):
    renter_mask = df['is_renter'] == 1
    denominator = (df['total_exp'] - df['spend_housing_kes']).clip(lower=1)  # avoid /0
    df['rent_burden_ratio'] = np.nan  # non-renters stay NaN by design
    df.loc[renter_mask, 'rent_burden_ratio'] = (
        df.loc[renter_mask, 'rent_actual_kes'] / denominator.loc[renter_mask]
    ).clip(upper=1.5)  # cap at 150% — extreme values are data errors

    df['rent_burdened'] = np.nan
    df.loc[renter_mask, 'rent_burdened'] = (
        df.loc[renter_mask, 'rent_burden_ratio'] > 0.30
    ).astype(float)

    n_burdened = df['rent_burdened'].eq(1).sum()
    n_renters_valid = renter_mask.sum()
    print(f'Renters with valid rent_burden_ratio: {renter_mask.sum():,}')
    print(f'Rent-burdened (>30% threshold, SDG 11): '
          f'{n_burdened:,} / {n_renters_valid:,} = '
          f'{n_burdened/n_renters_valid:.1%} of renters')

    # Cross-validate against stated burden (j09) and missed payments (j10)
    for val_col, label in [('housing_cost_burden', 'Stated burden (j09)'),
                            ('missed_payment', 'Missed payment (j10)')]:
        if val_col in df.columns:
            cross = pd.crosstab(
                df.loc[renter_mask, 'rent_burdened'],
                df.loc[renter_mask, val_col],
                normalize='index'
            )
            print(f'\n{label} × rent_burdened (row%):')
            print(cross.round(3).to_string())


In [ ]:
# ── 11.3  Crowding index: persons per room ────────────────────────────────────
if {'hh_size', 'dw_rooms'}.issubset(df.columns):
    # WHO threshold: >2 persons per room = overcrowded
    safe_rooms = df['dw_rooms'].replace(0, np.nan)
    df['persons_per_room'] = (df['hh_size'] / safe_rooms).clip(upper=15)
    df['is_overcrowded'] = (df['persons_per_room'] > 2).astype(int)
    print(f'persons_per_room constructed.')
    print(f'  Median: {df["persons_per_room"].median():.2f} persons/room')
    print(f'  Overcrowded (>2/room): {df["is_overcrowded"].sum():,} ({df["is_overcrowded"].mean():.1%})')
elif 'dw_bedrooms' in df.columns and 'hh_size' in df.columns:
    df['persons_per_room'] = (df['hh_size'] / df['dw_bedrooms'].replace(0, np.nan)).clip(upper=20)
    print('persons_per_room constructed from bedrooms (fallback).')


In [ ]:
# ── 11.4  Dwelling age ────────────────────────────────────────────────────────
if 'dwelling_yr_built' in df.columns:
    df['dw_age_yrs'] = 2024 - df['dwelling_yr_built']
    df['dw_age_yrs'] = df['dw_age_yrs'].clip(lower=0, upper=150)
    print(f'dw_age_yrs: median={df["dw_age_yrs"].median():.0f}yr, '
          f'max={df["dw_age_yrs"].max():.0f}yr, '
          f'missing={df["dw_age_yrs"].isna().sum()}')


In [ ]:
# ── 11.5  Objective quality score (Pillar 2) ─────────────────────────────────
# Materials scored 0 (worst) → 1 (best) following KHS distribution notes.
# dw_wall_mat: mud/wood=0, corrugated/wood=0.33, brick/block=0.67, stone/cement=1
# dw_roof_mat: grass/thatch=0, corrugated iron=0.5, concrete/tiles=1
# dw_floor_mat: earth=0, cement=0.5, tiles/wood=1
# Composite: equal-weighted average → 0 = fully unimproved, 1 = fully improved

# These mappings are based on KHS distribution notes in the Business Understanding doc.
# Validate against codebook before finalising pillar weights.

WALL_SCORE = {
    4: 0.00,   # mud/wood
    5: 0.00,   # mud block
    3: 0.25,   # timber/wood frame
    6: 0.25,   # corrugated metal
    2: 0.50,   # concrete block
    8: 0.50,   # burnt brick
    1: 0.75,   # stone/brick
    9: 0.75,   # precast concrete
   11: 0.75,   # burnt brick (variant)
   12: 1.00,   # stone/cement
    7: 1.00,   # reinforced concrete
   -1: np.nan,
}

ROOF_SCORE = {
    3: 0.00,   # grass/thatch
    4: 0.00,   # leaves/reeds
    2: 0.33,   # wood/timber
    5: 0.50,   # asbestos
    1: 0.67,   # iron sheets (most common: 45%)
    6: 0.83,   # tiles
    7: 1.00,   # concrete
    8: 0.67,   # iron sheets (variant)
   -1: np.nan,
}

FLOOR_SCORE = {
    3: 0.00,   # earth/mud
    4: 0.00,   # animal dung
    5: 0.25,   # wood planks
    1: 0.50,   # cement/concrete
    2: 1.00,   # tiles/terrazzo/marble
    6: 1.00,   # wood parquet
   -1: np.nan,
}

for col, score_map, name in [
    ('dw_wall_mat',  WALL_SCORE,  'wall_quality_score'),
    ('dw_roof_mat',  ROOF_SCORE,  'roof_quality_score'),
    ('dw_floor_mat', FLOOR_SCORE, 'floor_quality_score'),
]:
    if col in df.columns:
        df[name] = df[col].map(score_map)
        coverage = df[name].notna().mean()
        print(f'{name}: mean={df[name].mean():.3f}, coverage={coverage:.1%}')

score_components = [c for c in ['wall_quality_score', 'roof_quality_score',
                                  'floor_quality_score'] if c in df.columns]
if len(score_components) >= 2:
    df['obj_quality_score'] = df[score_components].mean(axis=1)
    print(f'\nobj_quality_score: mean={df["obj_quality_score"].mean():.3f}, '
          f'missing={df["obj_quality_score"].isna().sum()}')


In [ ]:
# ── 11.6  Triple-exposure flag (Pillar 4) ────────────────────────────────────
# triple_exposed = (flood_exposed > 0) AND (dw_type ≠ permanent) AND (no title_doc)
# This compound profile: highest-risk households invisible to single-indicator tools.

conditions_met = []

if 'flood_exposure' in df.columns:
    # flood_exposure: 0=no, 1=minor, 2=severe
    flood_exposed = df['flood_exposure'].gt(0).fillna(False)
    conditions_met.append(('flood_exposed', flood_exposed))

if 'dw_type' in df.columns:
    # dw_type: conventional/permanent=1, semi-permanent, temporary, traditional
    # Non-permanent = anything other than 1 (permanent) — verify against codebook
    informal_structure = df['dw_type'].ne(1).fillna(True)
    conditions_met.append(('informal_structure', informal_structure))

if 'has_title_doc' in df.columns:
    absent_title = df['has_title_doc'].ne(1).fillna(True)
    conditions_met.append(('absent_title', absent_title))
elif 'lp_has_title' in df.columns:
    absent_title = df['lp_has_title'].ne(1).fillna(True)
    conditions_met.append(('absent_title', absent_title))

if len(conditions_met) == 3:
    triple = conditions_met[0][1] & conditions_met[1][1] & conditions_met[2][1]
    df['triple_exposed'] = triple.astype(int)
    rate = df['triple_exposed'].mean()
    print(f'triple_exposed flag: {df["triple_exposed"].sum():,} households ({rate:.1%})')
    # County distribution (expected: Mombasa, Kisumu, Nairobi informal settlements)
    if 'county_code' in df.columns:
        cty_triple = df.groupby('county_code')['triple_exposed'].mean().sort_values(ascending=False)
        print('\nTop 10 counties by triple-exposure rate:')
        print(cty_triple.head(10).rename(COUNTY_MAP).to_string())
else:
    print(f'Only {len(conditions_met)} of 3 triple-exposure conditions available: '
          f'{[c[0] for c in conditions_met]}')


In [ ]:
# ── 11.7  Utility burden ratio ────────────────────────────────────────────────
# util_burden = (spend_water_kes + spend_electricity_kes) / total_exp
# Captures Pillar 5 (Utility Deprivation) financial dimension.

if {'spend_water_kes', 'spend_electricity_kes', 'total_exp'}.issubset(df.columns):
    util_spend = df['spend_water_kes'].fillna(0) + df['spend_electricity_kes'].fillna(0)
    df['util_burden_ratio'] = (util_spend / df['total_exp'].replace(0, np.nan)).clip(upper=1.0)
    print(f'util_burden_ratio: median={df["util_burden_ratio"].median():.3f}, '
          f'mean={df["util_burden_ratio"].mean():.3f}')

# ── 11.8  County name label column ───────────────────────────────────────────
if 'county_code' in df.columns:
    df['county_name'] = df['county_code'].map(COUNTY_MAP)
    unmapped = df['county_name'].isna().sum()
    print(f'county_name: {df["county_name"].nunique()} counties, '
          f'{unmapped} unmapped codes')
    print('Distribution:')
    print(df['county_name'].value_counts().head(10).to_string())


## 12 · Pillar Map Integrity Check & Leakage Assertion

All five HFVS pillars are mapped. A formal assertion confirms that no
formula ingredient appears in the proxy feature matrix (the anti-leakage
requirement from Business Understanding §5).

In [ ]:
# ── 12.1  Pillar variable map ────────────────────────────────────────────────

PILLAR_MAP = {
    'Pillar 1 – Financial Stress': [
        'total_exp', 'rent_burden_ratio', 'rent_burdened',
        'spend_food_kes', 'spend_clothing_kes', 'spend_education_kes',
        'spend_health_kes', 'spend_transport_kes', 'spend_comms_kes',
        'spend_recreation_kes', 'spend_housing_kes', 'spend_energy_kes',
        'spend_other_kes', 'spend_remittances_kes',
        'housing_cost_burden', 'missed_payment', 'eviction_risk',
        'rent_actual_kes', 'mort_rate', 'mort_no_market',
        'util_burden_ratio', 'loan_avg_kes', 'loan_outstanding_kes',
    ],
    'Pillar 2 – Physical Quality': [
        'dw_type', 'dw_wall_mat', 'dw_roof_mat', 'dw_floor_mat',
        'wall_quality_score', 'roof_quality_score', 'floor_quality_score',
        'obj_quality_score', 'dw_rooms', 'dw_area_m2', 'dw_bedrooms',
        'dw_approved', 'dw_has_planning', 'persons_per_room', 'is_overcrowded',
        'dw_age_yrs', 'perc_structure', 'perc_roof', 'perc_walls',
        'perc_floor', 'perc_ventilation', 'perc_overall',
        'water_src_main', 'toilet_type', 'cooking_fuel',
        'lighting_src', 'wsvc_water_conns', 'wsvc_sewer_conns',
    ],
    'Pillar 3 – Tenure Security': [
        'hh_tenure_type', 'tenure_type', 'is_owner_occupier', 'is_owner',
        'is_renter', 'has_title_doc', 'owns_land',
        'lp_n_parcels', 'lp_tenure_type', 'lp_has_title', 'lp_land_use',
        'lp_has_dispute', 'lp_is_registered', 'lp_used_as_collateral',
        'satisfied_tenure', 'yrs_in_dwelling', 'has_written_lease',
        'rent_arrears', 'eviction_risk', 'tenure_is_informal',
    ],
    'Pillar 4 – Physical Hazard Exposure': [
        'flood_exposure', 'landslide_exposure', 'other_hazard_exposure',
        'dw_in_hazard_zone', 'triple_exposed',
        'hazard_lost_property', 'hazard_displaced',
        'prob_poor_drainage',
    ],
    'Pillar 5 – Utility Deprivation': [
        'lighting_src', 'electricity_hrs_day', 'cooking_fuel',
        'water_src_main', 'water_treated', 'water_dist_mins',
        'toilet_type', 'has_handwash_facility',
        'spend_water_kes', 'spend_electricity_kes', 'spend_energy_other_kes',
        'util_income_ratio', 'cty_min_utility_kes', 'wsvc_tariff',
    ],
}

PROXY_FEATURES = {
    # Demographic
    'hh_size', 'hh_head_sex', 'has_disability', 'max_edu_isced',
    'mean_age', 'dependency_ratio', 'n_female', 'n_children',
    'n_elderly', 'n_working_age',
    # Geographic / admin
    'urban_rural', 'county_code', 'county_name', 'settlement_plan_status',
    # Assets
    'owns_mobile', 'owns_computer', 'owns_vehicle', 'has_internet',
    'owns_radio', 'owns_tv', 'owns_fridge',
    # Aspirations / revealed preference
    'wants_to_own', 'willing_to_relocate', 'plans_to_buy',
    'aware_affordable_housing', 'aspire_build', 'aspire_buy_house',
    # Stability
    'moved_last_5yrs' if 'moved_last_5yrs' in df.columns else None,
    # Market context (county-level — not HH-level formula ingredients)
    'nema_eia_apps', 'nema_eia_approvals', 'nema_proc_days',
    'mort_n_providers', 'loan_n_providers', 'fin_n_financiers',
}
PROXY_FEATURES = {f for f in PROXY_FEATURES if f is not None}

print('=== Pillar map coverage ===')
all_pillar_vars = set()
for pillar, cols in PILLAR_MAP.items():
    present  = [c for c in cols if c in df.columns]
    missing  = [c for c in cols if c not in df.columns]
    all_pillar_vars |= set(present)
    flag = f'  !! MISSING: {missing}' if missing else ''
    print(f'{pillar[:35]:<35} {len(present)}/{len(cols)} present{flag}')


In [ ]:
# ── 12.2  Anti-leakage assertion ─────────────────────────────────────────────
# No HFVS formula ingredient may appear in the proxy feature matrix.
# This is formally asserted here so any future changes to either set
# immediately fail the assertion.

leakage = all_pillar_vars & PROXY_FEATURES
if leakage:
    print(f'LEAKAGE WARNING: The following variables appear in BOTH '
          f'pillar map AND proxy matrix:')
    for v in sorted(leakage):
        print(f'  - {v}')
    raise AssertionError(
        f'{len(leakage)} leakage variable(s) detected! '
        f'Remove from proxy matrix before modelling.'
    )
else:
    print('Anti-leakage assertion PASSED. '
          'No overlap between pillar ingredients and proxy features. ✓')


## 13 · Save Cleaned Frame & Preprocessing Summary

In [ ]:
# ── 13.1  Final shape and remaining NaN check ────────────────────────────────
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(exclude=np.number).columns

print(f'Final cleaned shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Numeric columns      : {len(num_cols)}')
print(f'Categorical columns  : {len(cat_cols)}')

# Remaining NaN — should only be structural (KES continuous, lp_ for non-landowners etc.)
remaining_na = df.isna().mean().sort_values(ascending=False)
remaining_na = remaining_na[remaining_na > 0]
print(f'\nColumns still with NaN (structural by design): {len(remaining_na)}')
print(remaining_na.round(4).to_string())


In [ ]:
# ── 13.2  Save master_frame_clean.parquet ────────────────────────────────────
master_frame_clean = df.copy()
master_frame_clean.to_parquet(PQ / 'master_frame_clean.parquet', index=False)
print(f'master_frame_clean.parquet saved → {PQ / "master_frame_clean.parquet"}')

# Also save a weights-preserved summary table for county-level reporting
county_summary = (
    df.groupby('county_code')
    .agg(
        county_name   = ('county_name', 'first'),
        n_households  = ('hh_id', 'count'),
        n_renters     = ('is_renter', 'sum'),
        n_owners      = ('is_owner', 'sum'),
        med_total_exp = ('total_exp', 'median'),
        pct_rent_burdened = ('rent_burdened', 'mean'),
        pct_triple_exposed = ('triple_exposed', 'mean') if 'triple_exposed' in df.columns else ('hh_id', 'count'),
        mean_quality  = ('obj_quality_score', 'mean') if 'obj_quality_score' in df.columns else ('hh_id', 'count'),
        pct_overcrowded = ('is_overcrowded', 'mean') if 'is_overcrowded' in df.columns else ('hh_id', 'count'),
    )
    .reset_index()
)
county_summary.to_csv(TABS / 'county_profile_summary.csv', index=False)
print(f'county_profile_summary.csv saved → {TABS}')
county_summary.head()


In [ ]:
# ── 13.3  Preprocessing summary table (for report §3 / Phase 2 output) ───────

summary_rows = [
    ('KNBS sentinel codes decoded to NaN',
     sentinel_df["n_decoded_to_nan"].sum() if 'sentinel_df' in dir() else 'N/A',
     'Codes −1, 98, 99, 96, 998, 999, 996 → NaN (protected binary/ID cols exempt)'),

    ('Defensive column rename',
     len(rename_now),
     'Raw KNBS codes → semantic column names via RENAME_MAP'),

    ('Free-text "other/specify" fields dropped',
     len(free_text_cols) if 'free_text_cols' in dir() else 'N/A',
     'Unusable for quantitative analysis; near-100% missing'),

    ('Admin duplicate identifiers dropped',
     len(admin_drop) if 'admin_drop' in dir() else 'N/A',
     'Redundant with hh_id / county_code'),

    ('Constant columns dropped',
     len(constant_cols) if 'constant_cols' in dir() else 'N/A',
     'Zero variance — no analytical value'),

    ('Near-empty columns (>80% missing) dropped',
     len(near_empty_cols) if 'near_empty_cols' in dir() else 'N/A',
     'Insufficient data; outside protected core set'),

    ('Sparse multi-select sub-items dropped (<0.5%)',
     len(ms_prune_df) if 'ms_prune_df' in dir() else 'N/A',
     'Effectively constant; residual "other" tail options'),

    ('Rental k_* structural NaNs',
     len(k_cols) if 'k_cols' in dir() else 'N/A',
     'Recoded to −1 (Not applicable) for categorical; KES columns left NaN'),

    ('Owned l_* structural NaNs',
     len(l_cols) if 'l_cols' in dir() else 'N/A',
     'Recoded to −1 (Not applicable) for categorical; KES columns left NaN'),

    ('Land parcel lp_* structural NaNs (i00==0)',
     'lp_n_parcels + binary cols',
     'Zero-filled for landless households; lp_land_use/tenure → −1'),

    ('mort_rate structural NaN (no mortgage market)',
     'mort_rate',
     'Left NaN; mort_no_market binary flag created'),

    ('Remaining missing values imputed',
     len(impute_log_df) if 'impute_log_df' in dir() else 'N/A',
     'Group-median (by tenure type) for continuous; group-mode for ordinal'),

    ('Monetary columns Winsorized',
     len(outlier_df) if 'outlier_df' in dir() else 'N/A',
     'Negatives clipped to 0; high values capped at p99 of non-zero values'),

    ('Perception scale out-of-range values',
     'h01–h11',
     'Values outside {1,2,3} set to NaN'),

    ('Logical contradiction flags created',
     2,
     'flag_owner_with_rent, flag_nonrenter_with_rent'),

    ('Engineered features added',
     '9 new columns',
     'total_exp, rent_burden_ratio, rent_burdened, persons_per_room, dw_age_yrs, '
     'obj_quality_score, triple_exposed, util_burden_ratio, county_name'),
]

preprocessing_summary = pd.DataFrame(
    summary_rows, columns=['Issue / Action', 'Count / Scope', 'Treatment']
)
preprocessing_summary.to_csv(TABS / 'preprocessing_summary.csv', index=False)
print('Preprocessing summary saved.')
preprocessing_summary


## 14 · Quick Exploration of the Cleaned Frame

In [ ]:
# ── 14.1  Descriptive statistics — core numeric columns ──────────────────────
EXPLORE_COLS = [
    'total_exp', 'rent_burden_ratio', 'persons_per_room',
    'obj_quality_score', 'hh_size', 'dw_rooms', 'dw_bedrooms',
    'spend_housing_kes', 'rent_actual_kes', 'spend_electricity_kes',
    'spend_water_kes', 'water_dist_mins', 'yrs_in_dwelling',
    'util_burden_ratio', 'dependency_ratio', 'mean_age',
]
explore_cols_present = [c for c in EXPLORE_COLS if c in df.columns]

desc = df[explore_cols_present].agg([
    'count', 'mean', 'median', 'std',
    lambda x: x.max() - x.min(),
    lambda x: x.quantile(0.75) - x.quantile(0.25),
]).T
desc.columns = ['N', 'Mean', 'Median', 'Std Dev', 'Range', 'IQR']
print('Descriptive statistics — key analytical variables:')
desc.round(2)


In [ ]:
# ── 14.2  Urban vs Rural breakdown ───────────────────────────────────────────
if 'urban_rural' in df.columns:
    ur_map = {1: 'Urban', 2: 'Rural'}
    df['_ur_label'] = df['urban_rural'].map(ur_map)
    print('Urban / Rural household split:')
    print(df['_ur_label'].value_counts().to_string())

    if 'total_exp' in df.columns:
        print('\nMedian total_exp by residence:')
        print(df.groupby('_ur_label')['total_exp'].median().to_string())

    df = df.drop(columns=['_ur_label'])


In [ ]:
# ── 14.3  Tenure composition ──────────────────────────────────────────────────
if {'is_renter', 'is_owner'}.issubset(df.columns):
    tenure_comp = pd.DataFrame({
        'Renters'     : [df['is_renter'].sum(), f"{df['is_renter'].mean():.1%}"],
        'Owners'      : [df['is_owner'].sum(),  f"{df['is_owner'].mean():.1%}"],
        'Other/Informal': [
            ((df['is_renter']==0) & (df['is_owner']==0)).sum(),
            f"{((df['is_renter']==0) & (df['is_owner']==0)).mean():.1%}"
        ],
    }, index=['N', '%']).T
    print('Tenure composition:')
    print(tenure_comp.to_string())

if 'rent_burdened' in df.columns:
    n_rb = df['rent_burdened'].eq(1).sum()
    n_rtr = df['is_renter'].sum()
    print(f'\nRent-burdened (>30% threshold): {n_rb:,} / {n_rtr:,} renters = {n_rb/n_rtr:.1%}')


In [ ]:
# ── 14.4  Visualise key engineered features ──────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Cleaned Frame — Key Variable Distributions', fontsize=14, fontweight='600')

plot_pairs = [
    ('total_exp',         'Total Monthly Expenditure (KES)', TEAL),
    ('rent_burden_ratio', 'Rent Burden Ratio (Renters only)', RED),
    ('persons_per_room',  'Persons per Room (Crowding)', AMBER),
    ('obj_quality_score', 'Objective Quality Score (0–1)', BLUE),
    ('util_burden_ratio', 'Utility Burden Ratio', PURPLE),
    ('dependency_ratio',  'Dependency Ratio', GRAY),
]

for ax, (col, title, color) in zip(axes.flatten(), plot_pairs):
    if col not in df.columns:
        ax.set_visible(False)
        continue
    data = df[col].dropna()
    ax.hist(data, bins=40, color=color, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(data.median(), color='black', lw=1.5, linestyle='--', label=f'Median={data.median():.2f}')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGS / 'phase2_key_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved → phase2_key_distributions.png')


In [ ]:
# ── 14.5  County-level summary heatmap ───────────────────────────────────────
if 'county_name' in df.columns and 'total_exp' in df.columns:
    cty_stats = df.groupby('county_name').agg(
        med_exp        = ('total_exp', 'median'),
        pct_renter     = ('is_renter', 'mean'),
        pct_burdened   = ('rent_burdened', 'mean'),
        pct_overcrowded= ('is_overcrowded', 'mean') if 'is_overcrowded' in df.columns else ('hh_id', 'count'),
        mean_quality   = ('obj_quality_score', 'mean') if 'obj_quality_score' in df.columns else ('hh_id', 'count'),
    ).round(3)

    cty_stats_norm = (cty_stats - cty_stats.min()) / (cty_stats.max() - cty_stats.min() + 1e-9)
    cty_stats_norm = cty_stats_norm.sort_values('med_exp')

    fig, ax = plt.subplots(figsize=(12, 14))
    sns.heatmap(
        cty_stats_norm, ax=ax, cmap='RdYlGn_r',
        annot=False, linewidths=0.3, linecolor='white',
        xticklabels=['Med Exp', '% Renter', '% Burdened', '% Crowded', 'Quality Score'],
        cbar_kws={'label': 'Normalised value (0=best, 1=worst)', 'shrink': 0.5},
    )
    ax.set_title('County-Level Housing Indicators — Normalised Heatmap\n'
                 '(sorted by median expenditure, lowest → highest)',
                 fontsize=12, fontweight='600')
    ax.set_ylabel('County')
    plt.tight_layout()
    plt.savefig(FIGS / 'phase2_county_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('County heatmap saved → phase2_county_heatmap.png')


In [ ]:
# ── 14.6  Final audit: remaining NaN summary ─────────────────────────────────
final_na = df.isna().sum()
final_na = final_na[final_na > 0].sort_values(ascending=False)

print(f'=== Final cleaned frame ===')
print(f'Shape         : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory        : {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
print(f'Remaining NaN : {len(final_na)} columns (all structural by design)')
print()
print('Columns with NaN and their rates:')
na_pct = (df.isna().mean() * 100).loc[final_na.index].round(2)
na_summary = pd.DataFrame({'n_missing': final_na, 'pct_missing': na_pct})
print(na_summary.to_string())


---
## ✓ Phase 2 Complete

**Output:** `master_frame_clean.parquet` ready for Phase 3 (EDA) and HFVS construction.

**What was done:**
- KNBS sentinels decoded → NaN before any imputation
- Structural missingness (k_, l_, lp_*, mort_rate) correctly encoded (not imputed)
- Group-median imputation stratified by tenure type for remaining missingness
- KES variables Winsorized (p1–p99 of non-zero values only)
- 9 analytical features engineered: `total_exp`, `rent_burden_ratio`,
  `rent_burdened`, `persons_per_room`, `dw_age_yrs`, `obj_quality_score`,
  `triple_exposed`, `util_burden_ratio`, `county_name`
- Anti-leakage assertion confirmed: no pillar ingredient in proxy matrix
- Preprocessing summary table saved to `outputs/tables/dsa8301/`

**Next:** Phase 3 — EDA, normality gate, descriptive statistics per workflow.
